In [ ]:
data = FileAttachment("simulacion_10_seeds_susceptibility.json").json()

In [ ]:
run = data.find(d => d.W_params[0] === W_values[W_index] && d.W_params[1] === W_values[W_index])

In [ ]:
viewof W_index = Inputs.range([0, 3], {label: "W Index", step: 1, value: 0, format: i => W_values[i]})

In [ ]:
W_values = [0, 0.1, 0.5, 1.0]

In [ ]:
plot = {
  const width = 900;
  const height = 400;
  const margin = {top: 40, right: 30, bottom: 40, left: 50};

  // Asegúrate de que 'run' esté definido. 
  if (!run) return md`No data found for these parameters`;

  const svg = d3.create("svg").attr("viewBox", [0, 0, width, height]);
  
  const drawHistogram = (values, title, xOffset, maxY, showMean = false, fixedDomain = null) => {
      // Validación extra: si values no existe o está vacío, no dibujamos nada
      if (!values || values.length === 0) return;

      const plotWidth = (width / 2);
      
      const x = d3.scaleLinear()
        .domain(fixedDomain || d3.extent(values))
        .nice()
        .range([xOffset + margin.left, xOffset + plotWidth - margin.right]);
        
      const bins = d3.bin()
        .domain(x.domain())
        .thresholds(x.ticks(20))
        (values);
        
      const y = d3.scaleLinear()
        .domain([0, maxY])
        .nice()
        .range([height - margin.bottom, margin.top]);
        
      const g = svg.append("g");
      
      // X Axis con formato de 2 decimales
      g.append("g")
        .attr("transform", `translate(0,${height - margin.bottom})`)
        .call(d3.axisBottom(x).tickFormat(d => d.toFixed(2)));
        
      // Y Axis
      g.append("g")
        .attr("transform", `translate(${xOffset + margin.left},0)`)
        .call(d3.axisLeft(y));
        
      // Bars
      g.append("g")
        .attr("fill", "steelblue")
        .selectAll("rect")
        .data(bins)
        .join("rect")
        .attr("x", d => x(d.x0) + 1)
        .attr("width", d => Math.max(0, x(d.x1) - x(d.x0) - 1))
        .attr("y", d => y(d.length))
        .attr("height", d => y(0) - y(d.length));
        
      // Línea punteada del promedio
      if (showMean) {
        const mean = d3.mean(values);
        g.append("line")
          .attr("x1", x(mean))
          .attr("x2", x(mean))
          .attr("y1", margin.top)
          .attr("y2", height - margin.bottom)
          .attr("stroke", "red")
          .attr("stroke-width", 2)
          .attr("stroke-dasharray", "5,5");
          
        // Label del promedio
        g.append("text")
          .attr("x", x(mean))
          .attr("y", margin.top - 5)
          .attr("text-anchor", "middle")
          .style("fill", "red")
          .style("font-size", "12px")
          .text(`μ = ${mean.toFixed(2)}`);
      }
        
      // Title
      g.append("text")
        .attr("x", xOffset + (plotWidth / 2))
        .attr("y", margin.top / 2)
        .attr("text-anchor", "middle")
        .style("font-weight", "bold")
        .text(title);
  };
  
  // Verifica que las propiedades existan en 'run'
  if (run.donaciones_delta) {
      drawHistogram(run.donaciones_delta, "Donaciones Delta", 0, 500, true, [0, 1]);
  }
  // Fixed typo: susceptibilidaes -> susceptibilidades
  if (run.susceptibilidades) {
      drawHistogram(run.susceptibilidades, "Susceptibilidades", width / 2, 1000, false, [0, 0.5]);
  }
  
  return svg.node();
}

In [ ]:
lineChart = {
  const width = 900;
  const height = 300;
  const margin = {top: 20, right: 30, bottom: 30, left: 40};

  if (!run || !run.suscept_history) return md`No susceptibility history data found`;

  const svg = d3.create("svg")
      .attr("viewBox", [0, 0, width, height]);

  // Handle both 1D (averaged) and 2D (individual seeds) data
  let histories = run.suscept_history;
  let isAveraged = false;
  
  if (histories.length > 0 && typeof histories[0] === 'number') {
      histories = [histories]; // Wrap in array to treat as single line
      isAveraged = true;
  }

  const rounds = histories[0].length;
  
  const x = d3.scaleLinear()
    .domain([0, rounds - 1])
    .range([margin.left, width - margin.right]);

  const y = d3.scaleLinear()
    .domain([0, 0.5]) // Susceptibility is between 0 and 0.5
    .nice()
    .range([height - margin.bottom, margin.top]);

  const line = d3.line()
    .x((d, i) => x(i))
    .y(d => y(d));

  // X Axis
  svg.append("g")
    .attr("transform", `translate(0,${height - margin.bottom})`)
    .call(d3.axisBottom(x).ticks(width / 80).tickSizeOuter(0));

  // Y Axis
  svg.append("g")
    .attr("transform", `translate(${margin.left},0)`)
    .call(d3.axisLeft(y));

  // Draw lines for each seed
  svg.append("g")
      .attr("fill", "none")
      .attr("stroke", "steelblue")
      .attr("stroke-width", 1.5)
      .attr("stroke-opacity", isAveraged ? 1 : 0.3) 
    .selectAll("path")
    .data(histories)
    .join("path")
      .attr("d", d => line(d));

  // Add average line (only if we have multiple histories)
  if (!isAveraged && histories.length > 1) {
      const avgHistory = [];
      for(let i=0; i<rounds; i++) {
          let sum = 0;
          for(let j=0; j<histories.length; j++) {
              sum += histories[j][i];
          }
          avgHistory.push(sum / histories.length);
      }

      svg.append("path")
          .datum(avgHistory)
          .attr("fill", "none")
          .attr("stroke", "red")
          .attr("stroke-width", 2)
          .attr("d", line);
  }

  // Title
  svg.append("text")
      .attr("x", width / 2)
      .attr("y", margin.top)
      .attr("text-anchor", "middle")
      .style("font-weight", "bold")
      .text(isAveraged ? "Susceptibility Evolution (Average Only)" : "Susceptibility Evolution (All Seeds + Average)");

  return svg.node();
}